# Spice: attended free GPU media batch

Run one exported job while you are present. In Colab choose **Runtime → Change runtime type → GPU** using the free runtime. Availability, RAM and session duration vary. If no GPU is assigned, the worker returns a downloadable **pending** result with a blocker. There is no paid-provider fallback.

This notebook opens no public server or tunnel and installs no keepalive. It performs no real-time voice/video session. For live conversation use the phone's native offline voice or an ephemeral worker you explicitly supply.

Export on the phone with `python -m spicecore.freegpu export ...` (see `docs/free-gpu-media.md`), keep the original ZIP, upload it here, run the finite worker, download its result, then import against that original ZIP. Only explicitly approved fictional-persona images/videos belong in the job. Notebook upload sends those selected inputs to Colab; do not include secrets.


In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys

try:
    import torch
    GPU_AVAILABLE = torch.cuda.is_available()
    print({"cuda_available": GPU_AVAILABLE,
           "device": torch.cuda.get_device_name(0) if GPU_AVAILABLE else None,
           "free_gib": round(torch.cuda.mem_get_info()[0] / 1024**3, 1) if GPU_AVAILABLE else None})
except ImportError:
    GPU_AVAILABLE = False
    print("PyTorch not installed. Colab GPU runtimes normally provide it; GPU jobs will remain pending.")
print("No GPU is required for the selected Piper CPU voice. GPU-dependent jobs remain pending without CUDA.")


## Load your checked worker source

The default source is this project branch. Set a trusted repository URL and revision, or leave the URL blank and upload **exactly** `spicecore/freegpu.py`, `scripts/freegpu-worker.py` and `scripts/setup-musetalk-colab.py` from your checkout. Source upload works before a branch is published. The worker source is executable code: use your reviewed checkout, not an arbitrary job attachment. A fresh runtime is simplest if this setup was already run.


In [ ]:
from google.colab import files
import tempfile

REPO_URL = "https://github.com/jcltd303-hub/spice-hoes.git"
REPO_REF = "main"  # Or the reviewed commit containing this bridge.
ROOT = Path("/content/spice-freegpu-source")
if ROOT.exists():
    raise RuntimeError("Source directory already exists; use a fresh runtime or inspect it before reusing it.")
if REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True, timeout=180)
    if REPO_REF:
        subprocess.run(["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=ROOT, check=True, timeout=180)
        subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=ROOT, check=True, timeout=30)
else:
    previous = Path.cwd()
    with tempfile.TemporaryDirectory(prefix="spice-source-upload-", dir="/content") as staging:
        try:
            os.chdir(staging)
            source_upload = files.upload()
        finally:
            os.chdir(previous)
    expected = {"freegpu.py", "freegpu-worker.py", "setup-musetalk-colab.py"}
    if set(source_upload) != expected:
        raise ValueError("Upload exactly freegpu.py, freegpu-worker.py and setup-musetalk-colab.py from the checked checkout")
    (ROOT / "spicecore").mkdir(parents=True)
    (ROOT / "scripts").mkdir()
    (ROOT / "spicecore" / "__init__.py").write_text("")
    (ROOT / "spicecore" / "freegpu.py").write_bytes(source_upload["freegpu.py"])
    (ROOT / "scripts" / "freegpu-worker.py").write_bytes(source_upload["freegpu-worker.py"])
    (ROOT / "scripts" / "setup-musetalk-colab.py").write_bytes(source_upload["setup-musetalk-colab.py"])
    del source_upload
for relative in ("spicecore/freegpu.py", "scripts/freegpu-worker.py", "scripts/setup-musetalk-colab.py"):
    if not (ROOT / relative).is_file():
        raise RuntimeError("Selected repository revision does not contain " + relative)
sys.path.insert(0, str(ROOT))
from spicecore.freegpu import read_bundle
print("Worker source ready")


## Upload and validate one finite job

Upload the ZIP produced by the export command. The bridge verifies all paths, counts, sizes, hashes and options before extraction. It never executes commands from a job manifest.

For the common free T4 image job on the phone:

```bash
python -m spicecore.freegpu export --task video --pipeline svd-i2v --image /absolute/path/to/approved-fictional-persona.png --prompt "Scene descriptor, not SVD text guidance" --approved-fictional-persona --output svd-job.zip
```


In [ ]:
previous = Path.cwd()
with tempfile.TemporaryDirectory(prefix="spice-job-upload-", dir="/content") as staging:
    try:
        os.chdir(staging)
        uploaded = files.upload()
    finally:
        os.chdir(previous)
if len(uploaded) != 1 or not next(iter(uploaded)).lower().endswith(".zip"):
    raise ValueError("Upload exactly one exported job ZIP")
JOB_ZIP = Path("/content/spice-attended-job.zip")
if JOB_ZIP.exists():
    raise RuntimeError("A job is already loaded; use a fresh runtime for the next attended job")
JOB_ZIP.write_bytes(next(iter(uploaded.values())))
del uploaded
job = read_bundle(JOB_ZIP, kind="job")
TASK = job["task"]
print(json.dumps({"job_id": job["job_id"], "task": TASK, "options": job["options"]}, indent=2))


## Install only the selected task's inference dependencies

Colab supplies PyTorch/CUDA; this cell keeps that installation. Video uses Diffusers 0.35.1 and a compatible Transformers 4.x API. Faster-whisper needs CUDA 12 cuBLAS and cuDNN 9. If installation fails, resolve it in this attended runtime; no external API is substituted. For lip sync the next cell installs pinned MuseTalk 1.5 in a separate Python 3.10 environment, leaving Colab PyTorch intact.


In [ ]:
import site

packages = []
if TASK == "video":
    packages = ["diffusers==0.35.1", "transformers>=4.44,<5", "accelerate>=0.33,<2",
                "sentencepiece>=0.2,<1", "imageio-ffmpeg>=0.5,<1", "Pillow>=10,<12"]
elif TASK == "tts":
    packages = ["piper-tts>=1.3,<2"]
elif TASK == "transcribe":
    packages = ["faster-whisper>=1.1,<2"]
    # Keep existing compatible CUDA libraries rather than replacing torch's pins.
    roots = [Path(path) for path in site.getsitepackages()]
    if not any(list(root.glob("nvidia/cublas/lib/libcublas.so.12*")) for root in roots):
        packages.append("nvidia-cublas-cu12")
    if not any(list(root.glob("nvidia/cudnn/lib/libcudnn.so.9*")) for root in roots):
        packages.append("nvidia-cudnn-cu12>=9,<10")
if packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages], check=True, timeout=600)
if TASK in ("video", "transcribe", "lipsync"):
    subprocess.run(["apt-get", "-qq", "update"], check=True, timeout=180)
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True, timeout=180)
print("Selected dependencies installed; actual readiness is checked by the worker")


## Explicit Piper voice selection / isolated MuseTalk setup

For a TTS job this cell downloads the selected open-weights Piper voice once. Review that voice's model card/license and change the voice name if needed. It is generic TTS, not voice cloning. For a selected lip-sync job and an assigned GPU, this cell installs the upstream 1.5 dependencies and selected weights in an isolated environment. You can instead set an existing MuseTalk installation path. Failed setup remains an explicit blocker; no other service is substituted. Source revision and download locations are in the setup script.


In [ ]:
PIPER_VOICE = "en_US-lessac-medium"  # Explicit selection; see the voice model card.
PIPER_MODEL = None
MUSETALK_DIR = ""  # E.g. /content/MuseTalk, with all officially required weights installed.
MUSETALK_PYTHON = ""  # Optional path to that installation's Python executable.
INSTALL_MUSETALK = True  # Only used for a selected lip-sync job with CUDA.
if TASK == "lipsync" and GPU_AVAILABLE and INSTALL_MUSETALK and not MUSETALK_DIR:
    config = Path("/content/spice-musetalk-config.json")
    subprocess.run([sys.executable, str(ROOT / "scripts/setup-musetalk-colab.py"),
                    "--output-config", str(config)], check=True, timeout=2700)
    installed = json.loads(config.read_text())
    MUSETALK_DIR = installed["musetalk_dir"]
    MUSETALK_PYTHON = installed["musetalk_python"]
if TASK == "tts":
    voice_dir = Path("/content/spice-piper-voices")
    voice_dir.mkdir(exist_ok=True)
    subprocess.run([sys.executable, "-m", "piper.download_voices", PIPER_VOICE,
                    "--data-dir", str(voice_dir)], check=True, timeout=180)
    PIPER_MODEL = voice_dir / (PIPER_VOICE + ".onnx")
    if not PIPER_MODEL.is_file() or not Path(str(PIPER_MODEL) + ".json").is_file():
        raise RuntimeError("Piper voice download incomplete")


## Run once, serially, while attended

The manifest's timeout is at most 3600 seconds (default 1800); it includes weight loading and generation. The supervisor kills the inference process group at the deadline and exports pending with no artifacts. Standalone T2V defaults to CogVideoX-2b, FP16, sequential CPU offload, VAE slicing/tiling, 720×480, 17 frames at 8fps, 20 steps. For a free T4 I2V job export `--pipeline svd-i2v`: FP16 SVD-XT uses model CPU offload, UNet forward chunking and `decode_chunk_size=2`, 1024×576, 25 frames at 7fps, 25 steps. SVD follows the image and motion settings; its prompt is only a job descriptor and is not text guidance. CogVideoX 5B I2V is an explicit BF16 option and may remain pending on T4. The worker checks at least 4 GiB free GPU memory and 8 GiB available host RAM, and caps PyTorch video allocations at 14 GiB. These are guardrails, not a guarantee of fit or speed. Change models/options in the phone's export command; no automatic model substitution occurs.


In [ ]:
RESULT_ZIP = Path("/content") / ("spice-" + job["job_id"] + "-result.zip")
if RESULT_ZIP.exists():
    raise RuntimeError("A result already exists; download it before starting a new attended job")
command = [sys.executable, str(ROOT / "scripts/freegpu-worker.py"), str(JOB_ZIP),
           "--output", str(RESULT_ZIP), "--min-free-gpu-gib", "4",
           "--min-host-ram-gib", "8", "--max-gpu-gib", "14"]
if PIPER_MODEL:
    command += ["--piper-model", str(PIPER_MODEL)]
if MUSETALK_DIR:
    command += ["--musetalk-dir", MUSETALK_DIR]
if MUSETALK_PYTHON:
    command += ["--musetalk-python", MUSETALK_PYTHON]
worker_env = dict(os.environ)
# CTranslate2 discovers pip-installed CUDA libraries at child startup.
libraries = [str(root / relative) for root in map(Path, site.getsitepackages())
             for relative in ("nvidia/cublas/lib", "nvidia/cudnn/lib") if (root / relative).is_dir()]
if libraries:
    worker_env["LD_LIBRARY_PATH"] = os.pathsep.join(libraries + [worker_env.get("LD_LIBRARY_PATH", "")])
subprocess.run(command, cwd=ROOT, env=worker_env, check=True)
result = read_bundle(RESULT_ZIP, kind="result")
print(json.dumps({"status": result["status"], "cost_cents": result["cost_cents"],
                  "blocker": result["blocker"], "files": result["files"]}, indent=2))


## Download and import on the phone

Download completed, pending or failed results. Each job has a distinct result filename. For a MediaPipeline job, copy the downloaded ZIP into `data/freegpu/results/` and click **Retry render**; the controller imports it automatically. In Termux you can grant shared storage with `termux-setup-storage`, then `cp ~/storage/downloads/spice-*-result.zip data/freegpu/results/`. For a standalone job, keep the **original exported** job ZIP and import as below. Import validates matching job identity, task, source manifest and artifact hashes before creating local files:

```bash
python -m spicecore.freegpu import spice-attended-result.zip --job original-job.zip --output-dir data/freegpu/results
```

A pending result returns `artifact_paths: []`, `cost_cents: 0` and the blocker. Request compute again manually only when available. Completed output still needs your usual identity/quality/review/publishing gates. Disconnect/delete the runtime when finished.

Primary references: [SVD low-memory example](https://huggingface.co/docs/diffusers/v0.35.1/en/using-diffusers/svd), [SVD-XT model card](https://huggingface.co/stabilityai/stable-video-diffusion-img2vid-xt), [SVD-XT license source](https://huggingface.co/stabilityai/stable-video-diffusion-img2vid-xt/blob/main/LICENSE.md), [CogVideoX-2b model card](https://huggingface.co/zai-org/CogVideoX-2b), [Diffusers CogVideoX](https://huggingface.co/docs/diffusers/v0.35.1/en/api/pipelines/cogvideox), [Piper CLI](https://github.com/OHF-Voice/piper1-gpl/blob/main/docs/CLI.md), [faster-whisper CUDA requirements](https://github.com/SYSTRAN/faster-whisper#gpu), [MuseTalk installation](https://github.com/TMElyralab/MuseTalk#installation).


In [ ]:
files.download(str(RESULT_ZIP))
